In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
import torch

from google.colab import drive
from pycocotools.coco import COCO
from torch.utils.data import DataLoader

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/ucu-ai-hw-2")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.data_prep import load_annotations
from scripts.data_cv import stratified_group_folds
from scripts.dataset_helpers import DetectionDataset, collate_data
from scripts.model_builders import build_ssd_model
from scripts.experiment_results import model_training_results
from scripts.experiment_runner import run_torch_cv

DIRECTORY = PROJECT_ROOT / "data"
INFO_TRAIN = DIRECTORY / "train" / "annotations_coco.json"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
FOLDS = 3
EPOCHS = 5
FRAME_STEP = 15
BATCH_SIZE = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
train_frames, train_boxes = load_annotations(COCO(str(INFO_TRAIN)), split="train")
frames = stratified_group_folds(train_frames, train_boxes, n_splits=FOLDS, seed=SEED)
annotations_by_image = {image_id: group for image_id, group in train_boxes.groupby("image_id")}
empty_annotations = train_boxes.iloc[:0].copy()

loading annotations into memory...
Done (t=8.63s)
creating index...
index created!


In [ ]:
cv_fold_loaders = {}

kwargs_dataset = {
    "directory": DIRECTORY,
    "annotations_by_image": annotations_by_image,
    "empty_annotations": empty_annotations,
}

for fold in range(FOLDS):
    training_frames = (frames.loc[frames[f"train_fold_{fold}"].eq(True)].sort_values(["video", "frame_id"])
        .groupby("video", group_keys=False).nth[::FRAME_STEP].reset_index(drop=True))

    validation_frames = frames.loc[frames["validation_fold"].eq(fold)].reset_index(drop=True)

    train_dataset = DetectionDataset(frames=training_frames, validation=False, **kwargs_dataset)
    valid_dataset = DetectionDataset(frames=validation_frames, validation=True, **kwargs_dataset)

    cv_fold_loaders[fold] = {
        "train": DataLoader(train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=0,
            collate_fn=collate_data
        ),
        "valid": DataLoader(
            valid_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0,
            collate_fn=collate_data
        )
    }

    print(f"Fold {fold}: {len(train_dataset)} training frames, {len(valid_dataset)} validation frames")

Fold 0: 1107 training frames, 7943 validation frames
Fold 1: 1094 training frames, 8088 validation frames
Fold 2: 1091 training frames, 8170 validation frames


## SSD300 VGG16 baseline model 

In [ ]:
baseline_histories, baseline_results = run_torch_cv(cv_fold_loaders=cv_fold_loaders,
    model_method=build_ssd_model, output_dir=PROJECT_ROOT / "outputs" / "ssd_baseline",
    annotations_path=INFO_TRAIN, device=device, seed=SEED, epochs=EPOCHS)

loading annotations into memory...
Done (t=5.39s)
creating index...
index created!
Loading and preparing results...
DONE (t=6.09s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=362.50s).
Accumulating evaluation results...
DONE (t=13.68s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.024
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.060
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.019
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.003
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.022
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.064
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.006
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.026
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxD

In [ ]:
baseline_results_wrap = model_training_results(ROOT=PROJECT_ROOT, filename="ssd_baseline", cv_folds=range(FOLDS))


Fold 0


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,6.138452,0.0001,0.024498,0.060331,0.003320
1,2,4.704492,0.0001,0.024042,0.057136,0.004003
2,3,4.366478,0.0001,0.030370,0.070796,0.004938
3,4,4.089396,0.0001,0.033629,0.075736,0.004806
4,5,3.898374,0.0001,0.034114,0.079434,0.005917



Fold 1


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,6.387709,0.0001,0.029839,0.072360,0.004452
1,2,4.872934,0.0001,0.042306,0.092973,0.005231
2,3,4.426126,0.0001,0.044872,0.103823,0.004598
3,4,4.150155,0.0001,0.047395,0.111298,0.005999
4,5,3.944425,0.0001,0.048727,0.111486,0.005670



Fold 2


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,6.174944,0.0001,0.020394,0.052590,0.003968
1,2,4.788049,0.0001,0.025314,0.064523,0.005497
2,3,4.409291,0.0001,0.032313,0.081697,0.005677
3,4,4.151287,0.0001,0.035975,0.090161,0.006108
4,5,3.943019,0.0001,0.037497,0.091989,0.006412



Best results per fold


,fold,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,0,5,3.898374,0.0001,0.034114,0.079434,0.005917
1,1,5,3.944425,0.0001,0.048727,0.111486,0.005670
2,2,5,3.943019,0.0001,0.037497,0.091989,0.006412



Metrics mean and std across folds


,mAP,AP50,AP_small
mean,0.040113,0.094303,0.006000
std,0.007650,0.016151,0.000378
